# PI-CAI nnUNet v2 Professional Pipeline

**6-Channel Input**: T2W + ADC + HBV + 3 One-Hot Zonal Channels

## Two-Phase Workflow

| Phase | Purpose | Patients | Epochs |
|---|---|---|---|
| **Phase 1: Sanity Check** | Verify entire pipeline works end-to-end | 10 | 5 |
| **Phase 2: Full Training** | Production run | 1500 | 250 (default) |

**Run Phase 1 first!** Only proceed to Phase 2 after confirming everything works.

---

# ═══════════════════════════════════════════════════════════
# SHARED SETUP (Run these cells for BOTH phases)
# ═══════════════════════════════════════════════════════════

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ─── Verify dataset exists ───────────────────────────────
SOURCE_DATA = '/content/drive/MyDrive/PI-CAI_pre-processed'
assert os.path.exists(SOURCE_DATA), f'Dataset not found at {SOURCE_DATA}'

# Count files in each modality
for folder in ['t2', 'adc_reg', 'hbv_reg', 'zonal_masks', 'lesion_masks']:
    path = os.path.join(SOURCE_DATA, folder)
    if os.path.exists(path):
        count = len([f for f in os.listdir(path) if f.endswith('.nii.gz')])
        print(f'  {folder}: {count} files')
    else:
        print(f'  ⚠️ {folder}: NOT FOUND')

print(f'\n✅ Dataset verified at {SOURCE_DATA}')

## 2. Clone Repository & Install Dependencies

In [ ]:
import os

REPO_DIR = '/content/nnu-Net-Baseline'

if os.path.exists(REPO_DIR):
    print('Repository already cloned, pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    # ──────────────────────────────────────────────────────────
    # ⚠️ UPDATE THIS URL to your actual GitHub repository
    # ──────────────────────────────────────────────────────────
    !git clone https://github.com/HemishJain09/nnu-Net-Baseline.git {REPO_DIR}

print(f'\n✅ Repository ready')
!ls {REPO_DIR}/*.py

In [ ]:
# Clone original nnUNet repository and install from source (editable mode)
!rm -rf /content/nnUNet
!git clone https://github.com/MIC-DKFZ/nnUNet.git /content/nnUNet
!cd /content/nnUNet && pip install -e .
!pip install -q nibabel scikit-learn tqdm

# Verify installation
!nnUNetv2_plan_and_preprocess --help | head -3
print('\n✅ nnUNet v2 installed from source successfully')

## 3. Set Environment Variables

**Split Storage Architecture:**
- `nnUNet_raw` + `nnUNet_preprocessed` → Colab SSD (fast I/O)
- `nnUNet_results` → Google Drive (persistent checkpoints)

In [ ]:
import os

# Fast local SSD (ephemeral — wiped on disconnect)
os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'

# Persistent Google Drive (checkpoints survive disconnects)
RESULTS_DIR = '/content/drive/MyDrive/PI-CAI_nnUNet_Results'
os.environ['nnUNet_results'] = RESULTS_DIR

# Create all directories
for d in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed'], RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print('Environment Variables:')
print(f'  nnUNet_raw:          {os.environ["nnUNet_raw"]}          (local SSD)')
print(f'  nnUNet_preprocessed: {os.environ["nnUNet_preprocessed"]} (local SSD)')
print(f'  nnUNet_results:      {os.environ["nnUNet_results"]}      (Google Drive)')

# Show disk space
!echo '' && df -h /content | tail -1 | awk '{print "Local SSD: " $4 " free of " $2}'
print('\n✅ Environment configured')

# ═══════════════════════════════════════════════════════════
# PHASE 1: SANITY CHECK (10 patients, 5 epochs)
# ═══════════════════════════════════════════════════════════

Run this first to verify the full pipeline works before committing
your paid Colab Pro+ hours to the full 1500-patient training.

## Phase 1 — Step 1: Convert 10 Patients

In [ ]:
REPO_DIR = '/content/nnu-Net-Baseline'
SOURCE_DATA = '/content/drive/MyDrive/PI-CAI_pre-processed'

# Convert ONLY 10 patients for sanity check
!python {REPO_DIR}/convert_picai_to_nnunet.py \
    --source_dir {SOURCE_DATA} \
    --nnunet_raw $nnUNet_raw \
    --max_cases 10

In [ ]:
# ─── Verify conversion ──────────────────────────────────
import json, os

dataset_dir = os.path.join(os.environ['nnUNet_raw'], 'Dataset500_PICAI')
with open(os.path.join(dataset_dir, 'dataset.json')) as f:
    dj = json.load(f)

n_images = len(os.listdir(os.path.join(dataset_dir, 'imagesTr')))
n_labels = len(os.listdir(os.path.join(dataset_dir, 'labelsTr')))

print(f'Channels: {json.dumps(dj["channel_names"], indent=2)}')
print(f'Labels:   {dj["labels"]}')
print(f'\nImages: {n_images} files (expected: {dj["numTraining"]} × 6 = {dj["numTraining"] * 6})')
print(f'Labels: {n_labels} files (expected: {dj["numTraining"]})')

assert n_images == dj['numTraining'] * 6, f'❌ Image count mismatch! Got {n_images}, expected {dj["numTraining"] * 6}'
assert n_labels == dj['numTraining'], f'❌ Label count mismatch! Got {n_labels}, expected {dj["numTraining"]}'
print('\n✅ Data conversion verified!')

## Phase 1 — Step 2: nnUNet Plan & Preprocess

In [ ]:
# Run nnUNet's automated fingerprint extraction + preprocessing
# This verifies nnUNet v2 accepts our 6-channel data format
!nnUNetv2_plan_and_preprocess -d 500 --verify_dataset_integrity -c 3d_fullres

## Phase 1 — Step 3: Generate Patient-Level Splits

In [ ]:
# Step 4: Generate Domain Adaptation (Pure Holdout) Splits
# Upload your marksheet.csv to Colab (/content/marksheet.csv) before running this!
REPO_DIR = '/content/nnu-Net-Baseline'

import os
if not os.path.exists('/content/marksheet.csv'):
    raise FileNotFoundError("❌ marksheet.csv not found! Please upload it to /content/")

!python {REPO_DIR}/generate_splits.py \n    --nnunet_raw  \n    --nnunet_preprocessed  \n    --marksheet /content/marksheet.csv \n    --train_centers RUMC ZGT

## Phase 1 — Step 4: Train for 5 Epochs (Sanity Check)

We train for just 5 epochs to verify:
- Training loop starts without errors
- Loss decreases (model is learning)
- Checkpoint saves correctly to Google Drive
- GPU memory is sufficient

In [ ]:
# Modify the original nnUNet source code directly to run 5 epochs
trainer_file = '/content/nnUNet/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py'

!sed -i 's/self.num_epochs = 1000/self.num_epochs = 5/g' {trainer_file}
!grep 'self.num_epochs' {trainer_file}
print(f'\n✅ Modified nnUNetTrainer.py to run exactly 5 epochs for sanity check!
')


In [ ]:
# Run 5-epoch sanity check training on Fold 0
!nnUNetv2_train 500 3d_fullres 0

In [ ]:
# ─── Verify sanity check results ────────────────────────
import os

results_dir = os.environ['nnUNet_results']
print(f'Checking results directory: {results_dir}')
print()

# Walk through results to find checkpoints
found_checkpoint = False
for root, dirs, files in os.walk(results_dir):
    for f in files:
        if 'checkpoint' in f or 'training_log' in f or 'progress' in f:
            fpath = os.path.join(root, f)
            size_mb = os.path.getsize(fpath) / (1024*1024)
            print(f'  📄 {os.path.relpath(fpath, results_dir)} ({size_mb:.1f} MB)')
            if 'checkpoint' in f:
                found_checkpoint = True

if found_checkpoint:
    print('\n' + '='*60)
    print('✅ SANITY CHECK PASSED!')
    print('='*60)
    print('The pipeline works end-to-end:')
    print('  ✅ Data conversion (6 channels + blank masks)')
    print('  ✅ nnUNet preprocessing (fingerprint + resampling)')
    print('  ✅ Patient-level splits (no data leakage)')
    print('  ✅ Training runs & produces checkpoints')
    print('  ✅ Checkpoints saved to Google Drive')
    print('\n➡️  You can now proceed to PHASE 2 for full training!')
else:
    print('\n❌ No checkpoint found. Check the training output above for errors.')

# ═══════════════════════════════════════════════════════════
# PHASE 2: FULL TRAINING (1500 patients)
# ═══════════════════════════════════════════════════════════

**⚠️ Only run this after Phase 1 passes!**

This will:
1. Clean up sanity check data
2. Convert ALL 1500 patients
3. Preprocess the full dataset
4. Train with default 250 epochs

**Auto-Resume**: If Colab disconnects, re-run all cells from the top.
Cells 1-3 (setup) are idempotent. Phase 2 will detect existing checkpoints.

## Phase 2 — Step 1: Clean Up Sanity Check Data

In [ ]:
import shutil, os

# Remove sanity check data from local SSD
for d in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed']]:
    if os.path.exists(d):
        shutil.rmtree(d)
        os.makedirs(d)
        print(f'🧹 Cleaned: {d}')

# Also remove sanity check results from Drive
sanity_results = os.path.join(
    os.environ['nnUNet_results'],
    'Dataset500_PICAI',
    'nnUNetTrainer_5epochs__3d_fullres__nnUNetPlans'
)
if os.path.exists(sanity_results):
    shutil.rmtree(sanity_results)
    print(f'🧹 Cleaned sanity results from Drive')

print('\n✅ Ready for full dataset processing')

## Phase 2 — Step 2: Convert ALL 1500 Patients

This copies ~26GB from Drive to local SSD. Takes ~20-30 minutes.

In [ ]:
REPO_DIR = '/content/nnu-Net-Baseline'
SOURCE_DATA = '/content/drive/MyDrive/PI-CAI_pre-processed'

# Convert ALL patients (no --max_cases flag)
!python {REPO_DIR}/convert_picai_to_nnunet.py \
    --source_dir {SOURCE_DATA} \
    --nnunet_raw $nnUNet_raw

In [ ]:
# ─── Verify full conversion ─────────────────────────────
import json, os

dataset_dir = os.path.join(os.environ['nnUNet_raw'], 'Dataset500_PICAI')
with open(os.path.join(dataset_dir, 'dataset.json')) as f:
    dj = json.load(f)

n_images = len(os.listdir(os.path.join(dataset_dir, 'imagesTr')))
n_labels = len(os.listdir(os.path.join(dataset_dir, 'labelsTr')))

print(f'Total patients: {dj["numTraining"]}')
print(f'Images: {n_images} (expected: {dj["numTraining"] * 6})')
print(f'Labels: {n_labels} (expected: {dj["numTraining"]})')

assert n_images == dj['numTraining'] * 6
assert n_labels == dj['numTraining']
assert dj['numTraining'] == 1500, f'Expected 1500 patients, got {dj["numTraining"]}'
print('\n✅ All 1500 patients converted!')

## Phase 2 — Step 3: Plan & Preprocess (Full Dataset)

⏱️ This takes ~1-2 hours for 1500 patients.

In [ ]:
!nnUNetv2_plan_and_preprocess -d 500 --verify_dataset_integrity -c 3d_fullres

## Phase 2 — Step 4: Generate Patient-Level Splits

In [ ]:
# Step 4: Generate Domain Adaptation (Pure Holdout) Splits
# Upload your marksheet.csv to Colab (/content/marksheet.csv) before running this!
REPO_DIR = '/content/nnu-Net-Baseline'

import os
if not os.path.exists('/content/marksheet.csv'):
    raise FileNotFoundError("❌ marksheet.csv not found! Please upload it to /content/")

!python {REPO_DIR}/generate_splits.py \n    --nnunet_raw  \n    --nnunet_preprocessed  \n    --marksheet /content/marksheet.csv \n    --train_centers RUMC ZGT

## Phase 2 — Step 5: Train 3D Full-Resolution — Fold 0

Full training with default 250 epochs.

**Auto-Resume**: If Colab disconnects:
1. Re-run all cells from Cell 1 (setup cells are idempotent)
2. Skip Phase 1
3. Skip Phase 2 Step 1 (cleanup)
4. Re-run Phase 2 Steps 2-4 (conversion + preprocess)
5. This cell will detect checkpoint on Drive and resume with `--c`

**Expected Duration**: ~6-12 hours on Colab Pro+ (T4/A100)

In [ ]:
# Update the nnUNet source code to 250 epochs for faster production training
trainer_file = '/content/nnUNet/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py'
!sed -i 's/self.num_epochs = 5/self.num_epochs = 250/g' {trainer_file}
print(f'✅ Updated nnUNetTrainer.py to 250 epochs for faster production training!\n')

!nnUNetv2_train 500 3d_fullres 0


## (Optional) Train Remaining Folds

After Fold 0 completes, run remaining folds for full 5-fold cross-validation.
Each fold auto-resumes from checkpoint if Colab disconnects.

In [ ]:
# Uncomment one fold at a time:
# !nnUNetv2_train 500 3d_fullres 1
# !nnUNetv2_train 500 3d_fullres 2
# !nnUNetv2_train 500 3d_fullres 3
# !nnUNetv2_train 500 3d_fullres 4

## (Optional) Find Best Configuration & Run Inference

In [ ]:
# After all 5 folds complete:
# !nnUNetv2_find_best_configuration 500 -c 3d_fullres

# Run inference:
# !nnUNetv2_predict -i INPUT_FOLDER -o OUTPUT_FOLDER -d 500 -c 3d_fullres -f 0